# Controlled Task 5 detector alternatives for 16 November

Prepare two **local, unsubmitted** CSVs from the already saved Empirical Tail and LOF models on `eda_1_v2`. Keep v17's `id` and Tasks 1–4 exactly; set the top 6,531 anomaly scores to `1` for Task 5. Compare each eventual public result directly with v17. No model is fitted or retuned here.

In [1]:
from pathlib import Path
import sys, json, hashlib, joblib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline" / "save_model.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
import numpy as np
import pandas as pd
from src.pipeline.random_seed import set_seed
SEED = set_seed()
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
BASE = SRC / "submission/kaggle_t12345_t3log_t5_rankblend25_v17.csv"
base = pd.read_csv(BASE, dtype=str)
base_card = json.loads(BASE.with_suffix(".json").read_text(encoding="utf-8"))
assert base_card["output_sha256"] == sha(BASE)
sample = pd.read_csv(SRC / "submission/sample_submission.csv", usecols=["id"], dtype=str)
assert base.columns.tolist() == ["id", "task1", "task2", "task3", "task4", "task5"]
assert len(base) == len(sample) == 25889 and base.id.tolist() == sample.id.tolist()
assert base.id.is_unique and base.task5.astype(int).sum() == 6531
print("Frozen v17", sha(BASE), "alerts", base.task5.astype(int).sum())

Frozen v17 d4b7c7b32c8c051a3d398819523ddfde4373699a7c2c2537b30ed912b519a9c8 alerts 6531


In [2]:
data_card_path = SRC / "task5/data/eda/eda_1_v2.json"
data_card = json.loads(data_card_path.read_text(encoding="utf-8"))
assert set(data_card["splits"]) == {"test"}
frame = pd.read_parquet(SRC / data_card["splits"]["test"]["path"])
assert len(frame) == len(base) and frame.player_id.is_unique and "is_anomaly" not in frame
X = frame.drop(columns="player_id")
def load_saved(name):
    card_path = SRC / "task5/save" / f"{name}.json"
    card = json.loads(card_path.read_text(encoding="utf-8"))
    assert card["name"] == name and card["data_name"] == data_card["name"]
    return joblib.load(SRC / card["model_path"]), card_path

def fixed_budget_prediction(model, k=6531):
    scores = np.asarray(model.anomaly_score(X), dtype=float)
    assert scores.shape == (len(base),) and np.isfinite(scores).all()
    chosen = np.lexsort((np.arange(len(scores)), -scores))[:k]
    pred = np.zeros(len(scores), dtype=np.int8)
    pred[chosen] = 1
    assert int(pred.sum()) == k
    return pred

specs = [
    ("kaggle_t12345_t5_tail_engineered_v20", "t5_r2_tail_engineered_v2", 2885,
     "Empirical tail detector on v17's engineered features"),
    ("kaggle_t12345_t5_lof_engineered_v21", "t5_r2_lof_engineered_v2", 1405,
     "Local Outlier Factor on v17's engineered features"),
]
results = {}
for out_name, model_name, expected_swap, hypothesis in specs:
    model, model_card_path = load_saved(model_name)
    pred = fixed_budget_prediction(model)
    reference = base.task5.astype(np.int8).to_numpy()
    added = int(((pred == 1) & (reference == 0)).sum())
    removed = int(((pred == 0) & (reference == 1)).sum())
    assert added == removed == expected_swap
    results[out_name] = (pred, model_card_path, hypothesis, added)
    print(out_name, "overlap", 6531-added, "swap in/out", added)

kaggle_t12345_t5_tail_engineered_v20 overlap 3646 swap in/out 2885


kaggle_t12345_t5_lof_engineered_v21 overlap 5126 swap in/out 1405


In [3]:
for out_name, (pred, model_card_path, hypothesis, swap) in results.items():
    csv = SRC / "submission" / f"{out_name}.csv"
    meta = csv.with_suffix(".json")
    assert csv.exists() == meta.exists(), f"Incomplete destination: {out_name}"
    output = base.copy()
    output["task5"] = pred.astype(str)
    assert output[["id", "task1", "task2", "task3", "task4"]].equals(base[["id", "task1", "task2", "task3", "task4"]])
    assert output.notna().all().all() and output.task5.astype(int).sum() == 6531
    detail = {
        "name": out_name, "date_prepared": "2026-11-15", "planned_slot_date": "2026-11-16",
        "status": "prepared locally; not uploaded", "base_submission": BASE.name,
        "base_sha256": sha(BASE), "comparison_submission": BASE.name,
        "changed_task": "task5", "hypothesis": hypothesis,
        "score_formula": "saved model anomaly_score; select 6531 largest, row index breaks ties",
        "alert_budget": 6531, "alerts": int(pred.sum()),
        "swapped_in_vs_comparison": swap, "swapped_out_vs_comparison": swap,
        "task5_orientation": "1 = anomaly", "rows": len(output),
        "model_cards": [model_card_path.relative_to(SRC).as_posix()],
        "dataset_cards": [data_card_path.relative_to(SRC).as_posix()], "seed": SEED,
    }
    if csv.exists():
        saved = pd.read_csv(csv, dtype=str)
        existing = json.loads(meta.read_text(encoding="utf-8"))
        assert saved.equals(output) and existing["output_sha256"] == sha(csv)
        assert all(existing[k] == v for k, v in detail.items())
    else:
        output.to_csv(csv, index=False)
        detail["output_sha256"] = sha(csv)
        meta.write_text(json.dumps(detail, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("Verified", csv.name, "SHA-256", sha(csv))

Verified kaggle_t12345_t5_tail_engineered_v20.csv SHA-256 2dd79b546af197688d3ff35211cc744c2e86a28360ea3a01343f743364c1505a
Verified kaggle_t12345_t5_lof_engineered_v21.csv SHA-256 2bc372f7c2abaf71156166859dcf155aab4630f1d8ac8cb6b0f0e007616b7322
